# 📖 Lab 3: Video Processing Pipeline (Deep Dive)

**Non-functional requirement:** *The system should support low latency streaming, even in low bandwidth environments.*

To support adaptive bitrate streaming (Lab 2), the raw uploaded video must be **processed** into segments × formats + manifest files. This is compute-heavy, parallel work — modeled as a **DAG (Directed Acyclic Graph)**.

## 🏗️ Architecture — The Video Processing DAG

```
S3 (raw video)
    │
    │ S3 event notification
    v
┌──────────────────────────────────────────────────────────────────────────┐
│                    Video Processing Service (DAG)                        │
│                                                                          │
│  ┌───────────┐     ┌────────────────────────────────────────────────┐    │
│  │  Video     │     │  Per-segment parallel processing (fan-out):   │    │
│  │  Splitter  │────>│                                                │    │
│  │  (ffmpeg)  │     │  Seg 1 ──> Transcode 1080p ──┐               │    │
│  └───────────┘     │         ──> Transcode 720p  ──┤               │    │
│                     │         ──> Transcode 480p  ──┤  fan-in       │    │
│                     │         ──> Transcode 360p  ──┤───────┐       │    │
│                     │         ──> Audio processing ──┤       │       │    │
│                     │         ──> Transcript gen.  ──┘       │       │    │
│                     │                                        v       │    │
│                     │  Seg 2 ──> (same)                ┌─────────┐  │    │
│                     │  Seg 3 ──> (same)                │ Build   │  │    │
│                     │  ...                             │manifests│  │    │
│                     └──────────────────────────────────│ + mark  │  │    │
│                                                        │ "done"  │  │    │
│                                                        └─────────┘  │    │
└──────────────────────────────────────────────────────────────────────────┘
    │                                                        │
    v                                                        v
S3 (segments + manifests)                          Metadata DB (status: ready)
```

## Learning Objectives

- Understand why video processing is a DAG (dependencies + parallelism)
- Simulate the full pipeline: split → transcode → manifest → mark done
- See fan-out parallelism in action (N segments × M formats)
- Compare sequential vs parallel processing
- Understand how orchestrators (Temporal, Airflow) coordinate this work

## 🛠️ Setup

```bash
cd system-designs/youtube
docker-compose up -d
```

Select the **"YouTube (Python)"** kernel.

In [ ]:
import psycopg2
import psycopg2.extras
from minio import Minio
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import os
import io
import json
import uuid

DB_CONFIG = {
    "host": "localhost", "port": 5434,
    "user": "demo", "password": "demo", "database": "youtube",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

minio_client = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)

# Upload a fake "raw video" to process
VIDEO_ID = uuid.uuid4().hex[:12]
RAW_SIZE = 10 * 1024 * 1024  # 10 MB simulated video
raw_data = os.urandom(RAW_SIZE)

minio_client.put_object("raw-videos", f"{VIDEO_ID}/original.mp4", io.BytesIO(raw_data), length=RAW_SIZE)

conn = get_connection()
cur = conn.cursor()
cur.execute("""
    INSERT INTO videos (id, title, description, user_id, status, content_type, file_size_bytes, raw_url)
    VALUES (%s, 'Processing Demo Video', 'Testing the pipeline', 1, 'uploaded', 'video/mp4', %s, %s)
""", (VIDEO_ID, RAW_SIZE, f"raw-videos/{VIDEO_ID}/original.mp4"))
conn.commit()
cur.close()
conn.close()

print(f"✅ Raw video uploaded: {VIDEO_ID} ({RAW_SIZE / 1024 / 1024:.0f}MB)")
print(f"✅ MinIO: {[b.name for b in minio_client.list_buckets()]}")

## 📊 Understanding the DAG

A DAG (Directed Acyclic Graph) is a graph of tasks where:
- **Directed** — each edge represents a dependency (A must finish before B starts)
- **Acyclic** — no circular dependencies (can't loop)

Our video processing DAG:

```
                    ┌─> transcode(seg1, 1080p) ─┐
                    ├─> transcode(seg1, 720p)  ──┤
         ┌─ seg1 ──┤─> transcode(seg1, 480p)  ──┼─┐
         │         ├─> transcode(seg1, 360p)  ──┤  │
         │         └─> audio(seg1)            ──┘  │
split ───┤                                         ├──> build_manifests ──> mark_done
         │         ┌─> transcode(seg2, 1080p) ─┐  │
         │         ├─> transcode(seg2, 720p)  ──┤  │
         ├─ seg2 ──┤─> transcode(seg2, 480p)  ──┼─┤
         │         ├─> transcode(seg2, 360p)  ──┤  │
         │         └─> audio(seg2)            ──┘  │
         │                                         │
         └─ segN ──┤─> ... (same)              ────┘
```

**Fan-out:** 1 video → N segments × M formats = N×M parallel tasks  
**Fan-in:** All transcoded segments → build manifest files → mark done

For a 1-hour video with 5s segments = 720 segments × 4 formats = **2,880 parallel transcode tasks**. This is why we need distributed worker nodes — impossible on a single machine.

Let's build each stage.

## Stage 1: Video Splitter

Takes the raw video and splits it into fixed-duration segments. In production this uses `ffmpeg`; we simulate by splitting the bytes.

In [ ]:
SEGMENT_DURATION = 5  # seconds
TOTAL_DURATION = 30   # simulated video length
NUM_SEGMENTS = TOTAL_DURATION // SEGMENT_DURATION

QUALITIES = {
    "1080p": {"bandwidth": 5000000, "resolution": "1920x1080", "size_ratio": 1.0},
    "720p":  {"bandwidth": 2500000, "resolution": "1280x720",  "size_ratio": 0.5},
    "480p":  {"bandwidth": 1000000, "resolution": "854x480",   "size_ratio": 0.2},
    "360p":  {"bandwidth": 500000,  "resolution": "640x360",   "size_ratio": 0.1},
}

BUCKET = "processed-videos"


def stage_split(video_id: str) -> list[str]:
    """
    Stage 1: Split the raw video into segments.
    
    In production: ffmpeg -i input.mp4 -c copy -segment_time 5 -f segment seg_%03d.ts
    Here: we split the raw bytes into equal chunks and store in S3 as temp files.
    """
    print(f"  📌 Stage 1: SPLIT — splitting into {NUM_SEGMENTS} segments")
    start = time.time()

    # Download raw video
    obj = minio_client.get_object("raw-videos", f"{video_id}/original.mp4")
    raw = obj.read()
    obj.close()

    chunk_size = len(raw) // NUM_SEGMENTS
    segment_keys = []

    for i in range(NUM_SEGMENTS):
        segment_data = raw[i * chunk_size : (i + 1) * chunk_size]
        key = f"{video_id}/temp/raw_segment_{i:03d}.ts"
        minio_client.put_object(BUCKET, key, io.BytesIO(segment_data), length=len(segment_data))
        segment_keys.append(key)

    ms = (time.time() - start) * 1000
    print(f"     → {NUM_SEGMENTS} segments stored in S3 ({ms:.0f}ms)")

    return segment_keys


segments = stage_split(VIDEO_ID)
for s in segments[:3]:
    print(f"     {s}")
print(f"     ...")

## Stage 2: Transcode (Fan-Out — Parallel)

Each segment is transcoded into multiple formats. This is the most CPU-intensive step. In production, each transcode task runs on a separate worker node.

We'll compare **sequential** vs **parallel** to see why parallelism matters.

In [ ]:
def transcode_segment(video_id: str, segment_key: str, segment_num: int, quality: str, info: dict) -> dict:
    """
    Transcode one segment into one quality level.
    
    In production: ffmpeg -i seg_001.ts -c:v libx264 -b:v 2500k -s 1280x720 seg_001_720p.ts
    Here: we simulate by reading the segment, creating a proportionally-sized output.
    """
    # Read raw segment from S3
    obj = minio_client.get_object(BUCKET, segment_key)
    raw_segment = obj.read()
    obj.close()

    # Simulate transcoding: resize to proportion (CPU-bound in production)
    transcoded_size = int(len(raw_segment) * info["size_ratio"])
    transcoded_data = os.urandom(transcoded_size)

    # Simulate CPU work (transcoding is slow)
    time.sleep(0.05)

    # Store transcoded segment in S3
    output_key = f"{video_id}/{quality}/segment_{segment_num:03d}.ts"
    minio_client.put_object(BUCKET, output_key, io.BytesIO(transcoded_data), length=transcoded_size)

    return {
        "segment": segment_num,
        "quality": quality,
        "input_size": len(raw_segment),
        "output_size": transcoded_size,
        "output_key": output_key,
    }


# ── Sequential processing ──
print("📌 Stage 2a: Transcode SEQUENTIALLY\n")
start_seq = time.time()

seq_results = []
total_tasks = NUM_SEGMENTS * len(QUALITIES)
task_num = 0

for seg_num, seg_key in enumerate(segments):
    for quality, info in QUALITIES.items():
        result = transcode_segment(VIDEO_ID, seg_key, seg_num, quality, info)
        seq_results.append(result)
        task_num += 1

seq_time = (time.time() - start_seq) * 1000
print(f"  {total_tasks} transcode tasks completed in {seq_time:.0f}ms (sequential)\n")


# ── Parallel processing ──
print("📌 Stage 2b: Transcode IN PARALLEL\n")
start_par = time.time()

par_results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = []
    for seg_num, seg_key in enumerate(segments):
        for quality, info in QUALITIES.items():
            futures.append(executor.submit(transcode_segment, VIDEO_ID, seg_key, seg_num, quality, info))

    for f in as_completed(futures):
        par_results.append(f.result())

par_time = (time.time() - start_par) * 1000
print(f"  {total_tasks} transcode tasks completed in {par_time:.0f}ms (8 parallel workers)\n")

speedup = seq_time / par_time if par_time > 0 else float("inf")
print(f"  📊 Speedup: {speedup:.1f}x faster with parallelism!")
print(f"     Sequential: {seq_time:.0f}ms")
print(f"     Parallel:   {par_time:.0f}ms")
print(f"\n  💡 In production with 720 segments × 4 formats = 2,880 tasks,")
print(f"     distributed across 100 worker nodes, this stage takes seconds instead of hours.")

## Stage 3: Build Manifests (Fan-In)

All transcoding is done. Now we generate HLS manifest files — the "index" that the client reads to know which segments are available in which quality.

In [ ]:
def stage_build_manifests(video_id: str) -> str:
    """
    Stage 3: Build HLS manifest files from the transcoded segments.
    Produces one master manifest + one media manifest per quality.
    """
    print(f"  📌 Stage 3: BUILD MANIFESTS")
    base_url = f"http://localhost:9000/{BUCKET}/{video_id}"

    # Media manifests (one per quality)
    for quality in QUALITIES:
        lines = [
            "#EXTM3U", "#EXT-X-VERSION:3",
            f"#EXT-X-TARGETDURATION:{SEGMENT_DURATION}",
            "#EXT-X-MEDIA-SEQUENCE:0",
        ]
        for seg_num in range(NUM_SEGMENTS):
            lines.append(f"#EXTINF:{SEGMENT_DURATION}.000,")
            lines.append(f"{base_url}/{quality}/segment_{seg_num:03d}.ts")
        lines.append("#EXT-X-ENDLIST")

        content = "\n".join(lines)
        minio_client.put_object(
            BUCKET, f"{video_id}/{quality}.m3u8",
            io.BytesIO(content.encode()), length=len(content),
            content_type="application/vnd.apple.mpegurl",
        )

    # Master manifest
    master_lines = ["#EXTM3U"]
    for quality, info in QUALITIES.items():
        master_lines.append(f'#EXT-X-STREAM-INF:BANDWIDTH={info["bandwidth"]},RESOLUTION={info["resolution"]}')
        master_lines.append(f"{base_url}/{quality}.m3u8")

    master_content = "\n".join(master_lines)
    minio_client.put_object(
        BUCKET, f"{video_id}/master.m3u8",
        io.BytesIO(master_content.encode()), length=len(master_content),
        content_type="application/vnd.apple.mpegurl",
    )

    manifest_url = f"{base_url}/master.m3u8"
    print(f"     → Master manifest: {manifest_url}")
    print(f"     → {len(QUALITIES)} media manifests created")

    return manifest_url


manifest_url = stage_build_manifests(VIDEO_ID)

## Stage 4: Mark Done

The final step: update the metadata DB to mark the video as `ready` and store the manifest URL. After this, the video is available for streaming.

In [ ]:
def stage_mark_done(video_id: str, manifest_url: str):
    """Stage 4: Update metadata DB — video is ready for streaming."""
    print(f"  📌 Stage 4: MARK DONE")

    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        UPDATE videos SET status = 'ready', manifest_url = %s, updated_at = NOW()
        WHERE id = %s
    """, (manifest_url, video_id))

    # Also record the format details
    for quality, info in QUALITIES.items():
        cur.execute("""
            INSERT INTO video_formats (video_id, resolution, codec, container, bitrate_kbps, segment_count)
            VALUES (%s, %s, 'h264', 'mpegts', %s, %s)
        """, (video_id, quality, info["bandwidth"] // 1000, NUM_SEGMENTS))

    conn.commit()
    cur.close()
    conn.close()
    print(f"     → Video {video_id} status: ready")
    print(f"     → {len(QUALITIES)} format records saved")


stage_mark_done(VIDEO_ID, manifest_url)

# Verify
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, title, status, manifest_url FROM videos WHERE id = %s", (VIDEO_ID,))
v = cur.fetchone()
print(f"\n  📊 Final video state:")
print(f"     id: {v['id']}")
print(f"     title: {v['title']}")
print(f"     status: {v['status']}")
print(f"     manifest: {v['manifest_url']}")

cur.execute("SELECT resolution, codec, container, bitrate_kbps, segment_count FROM video_formats WHERE video_id = %s", (VIDEO_ID,))
formats = cur.fetchall()
print(f"\n  📊 Available formats:")
for f in formats:
    print(f"     {f['resolution']} — {f['codec']}/{f['container']} @ {f['bitrate_kbps']}kbps ({f['segment_count']} segments)")
cur.close()
conn.close()

## 📊 The Full Pipeline: End-to-End

Let's see the complete pipeline from raw upload to streamable video, with timing for each stage.

In [ ]:
def run_full_pipeline(video_id: str) -> dict:
    """Run the complete video processing pipeline as a DAG."""
    timings = {}
    pipeline_start = time.time()

    # Stage 1: Split
    t = time.time()
    seg_keys = stage_split(video_id)
    timings["split"] = round((time.time() - t) * 1000)

    # Stage 2: Transcode (parallel)
    t = time.time()
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = []
        for seg_num, seg_key in enumerate(seg_keys):
            for quality, info in QUALITIES.items():
                futures.append(executor.submit(transcode_segment, video_id, seg_key, seg_num, quality, info))
        for f in as_completed(futures):
            f.result()
    timings["transcode"] = round((time.time() - t) * 1000)

    # Stage 3: Build manifests
    t = time.time()
    m_url = stage_build_manifests(video_id)
    timings["manifests"] = round((time.time() - t) * 1000)

    # Stage 4: Mark done
    t = time.time()
    stage_mark_done(video_id, m_url)
    timings["mark_done"] = round((time.time() - t) * 1000)

    timings["total"] = round((time.time() - pipeline_start) * 1000)
    return timings


# Process a fresh video through the full pipeline
FRESH_ID = uuid.uuid4().hex[:12]
fresh_data = os.urandom(10 * 1024 * 1024)
minio_client.put_object("raw-videos", f"{FRESH_ID}/original.mp4", io.BytesIO(fresh_data), length=len(fresh_data))
conn = get_connection()
cur = conn.cursor()
cur.execute("INSERT INTO videos (id, title, user_id, status, raw_url) VALUES (%s, 'Pipeline Demo', 1, 'uploaded', %s)",
            (FRESH_ID, f"raw-videos/{FRESH_ID}/original.mp4"))
conn.commit()
cur.close()
conn.close()

print(f"🎬 Running full pipeline for video {FRESH_ID}:\n")
timings = run_full_pipeline(FRESH_ID)

print(f"\n{'='*50}")
print(f"📊 Pipeline Timing Summary")
print(f"{'='*50}")
print(f"  Stage 1 — Split:         {timings['split']:>6}ms")
print(f"  Stage 2 — Transcode:     {timings['transcode']:>6}ms  (parallel, {NUM_SEGMENTS * len(QUALITIES)} tasks)")
print(f"  Stage 3 — Manifests:     {timings['manifests']:>6}ms")
print(f"  Stage 4 — Mark done:     {timings['mark_done']:>6}ms")
print(f"  {'─'*40}")
print(f"  Total pipeline:          {timings['total']:>6}ms")

# Count objects in S3
objects = list(minio_client.list_objects(BUCKET, prefix=f"{FRESH_ID}/", recursive=True))
print(f"\n  📦 S3 objects created: {len(objects)}")
print(f"     - {NUM_SEGMENTS * len(QUALITIES)} transcoded segments")
print(f"     - {len(QUALITIES)} media manifests + 1 master manifest")
print(f"     - {NUM_SEGMENTS} temp raw segments")

## 🧹 Cleanup

In [ ]:
from minio.deleteobjects import DeleteObject

# Clean up test videos from DB
conn = get_connection()
cur = conn.cursor()
cur.execute("DELETE FROM video_formats WHERE video_id IN (%s, %s)", (VIDEO_ID, FRESH_ID))
cur.execute("DELETE FROM videos WHERE id IN (%s, %s)", (VIDEO_ID, FRESH_ID))
conn.commit()
cur.close()
conn.close()

# Clean up S3
for vid in [VIDEO_ID, FRESH_ID]:
    for bucket in ["raw-videos", "processed-videos"]:
        objects = list(minio_client.list_objects(bucket, prefix=f"{vid}/", recursive=True))
        if objects:
            delete_list = [DeleteObject(obj.object_name) for obj in objects]
            list(minio_client.remove_objects(bucket, delete_list))

print("✅ Cleaned up test data from PostgreSQL and S3.")

## ✅ Summary

### The Video Processing DAG

| Stage | Input | Output | Parallelism |
|-------|-------|--------|-------------|
| **1. Split** | Raw video (1 file) | N raw segments | Sequential |
| **2. Transcode** | N raw segments | N × M transcoded segments | **Embarrassingly parallel** — each task is independent |
| **3. Manifests** | Segment file list | Master + M media manifests | Sequential (fast — just text files) |
| **4. Mark done** | Manifest URL | DB update (status: ready) | Sequential (single DB write) |

### Key Design Decisions

| Decision | Why |
|----------|-----|
| **DAG model** | Clear dependency graph — split before transcode, transcode before manifests |
| **Fan-out parallelism** | N segments × M formats = thousands of independent CPU-bound tasks |
| **S3 for temp data** | Workers pass URLs, not files — decouples workers from each other |
| **Orchestrator (Temporal/Airflow)** | Handles scheduling, retries, dependency tracking, worker assignment |
| **Segments, not whole files** | Enables adaptive bitrate streaming, seeking, and partial playback |

### Production Scale

```
1-hour video @ 5s segments:
  720 segments × 4 formats = 2,880 transcode tasks
  + 720 audio tasks + 720 transcript tasks = ~4,320 total tasks

With 100 worker nodes:
  ~43 tasks per worker → completes in seconds

Without parallelism:
  ~43 tasks × 50ms each × 4,320 = ~216 seconds (3.6 minutes) on one machine
```

### Interview Guidance

- You don't need to be an ffmpeg expert — just know the pipeline stages and their dependencies
- The key insight is **fan-out parallelism** — transcoding is CPU-bound and embarrassingly parallel
- Mention a DAG orchestrator (Temporal, Airflow, Step Functions) — don't build your own scheduler
- S3 as the interchange format between workers — avoids passing large files between services